## Define libraries

In [1]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from helper import RAGHelper
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
import os
from datasets import load_dataset

import sqlite3
import pandas as pd
import re
import json

import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score

In [3]:
# ingestion
# embedding_model - 2
# chunking_size - 256, 512, 1024
# chunking_overlap - 200, 100, 50
# search_type - 2
# search_k - 3, 5, 7
# vector_database - 3
# retrieval techniques - 2
# reranking
# gen_model - 2



## Define Variables

In [4]:
# Constants

DATASET_SOURCE = 'rungalileo/ragbench'
DATASET_NAME = 'finqa'
DATA_SPLIT = 'test'
VECTOR_DATABASE = 'chroma'
GROQ_API_KEY = "GROQ_API_KEY"
TABLE_NAME = 'nextgenrag_v1'
DATABASE_URL = os.getenv('DATABASE_URL')
DOMAIN = 'finance'







In [5]:
### Parameters

chunking_size = 1024
chunking_overlap = 200
separators = ["\n\n", "\n", " ", ".", ","]
# embedding
chromadb_folder = "../database"
gen_model = "llama-3.1-8b-instant"
embedding_model = "BAAI/bge-base-en-v1.5"

db_name = f"finance_{chunking_size}_{chunking_overlap}"
persist_directory = f"{chromadb_folder}/{db_name}"

search_type = "similarity"
search_kwargs = {"k":3}


## Data Fetching

In [6]:

dataset = load_dataset(DATASET_SOURCE, DATASET_NAME, split=DATA_SPLIT)

In [7]:
def deduplicate_data(data):
    data_dict = {}
    for d in data:
        # print(d)
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            # print(d)
            # break
            data_dict[document] = {"docid":[d["id"]]}
    return data_dict

In [8]:
dedup = deduplicate_data(dataset)

In [9]:
docs = [
    Document(
        
            metadata=v, 
            page_content=k
        
        
    )
    for k,v in dedup.items()
]

## Data Ingestion

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunking_size, chunk_overlap=chunking_overlap, separators=separators)
docs_chunks = text_splitter.split_documents(docs)

In [11]:
if os.path.exists(persist_directory) and os.listdir(persist_directory):
    print("Loading existing vector database...")
    vector_db = Chroma(
        persist_directory=persist_directory, 
        embedding_function=HuggingFaceEmbeddings(model_name=embedding_model)
    )
else:
    print("Creating new vector database...")
    # This assumes you have 'docs_chunks' already defined as per your notebook
    vector_db = Chroma.from_documents(
        documents=docs_chunks, 
        embedding=HuggingFaceEmbeddings(model_name=embedding_model), 
        persist_directory=persist_directory
    )

Loading existing vector database...


/var/folders/3f/z4mxt4h16g95cy326q6m_cqm0000gn/T/ipykernel_93812/3160788537.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function=HuggingFaceEmbeddings(model_name=embedding_model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Data Retrieval & Inference

In [12]:
api_key = os.getenv(GROQ_API_KEY)
rg = RAGHelper(api_key=api_key)


In [13]:
retriever = vector_db.as_retriever(search_type=search_type, search_kwargs=search_kwargs)

In [14]:
import time

start = time.time()
rg.gen_db_insert(dataset=dataset, retriever=retriever,
              chunk_size=chunking_size,
              chunk_overlap=chunking_overlap,
              table_name=TABLE_NAME, database_url=DATABASE_URL,
              domain=DOMAIN, vector_db=VECTOR_DATABASE,
              embed_model=embedding_model,  gen_model=gen_model, temperature=0)

print("Time:", time.time() - start)

1/1138 completed
2/1138 completed
3/1138 completed
4/1138 completed
5/1138 completed
6/1138 completed
7/1138 completed
8/1138 completed
9/1138 completed
10/1138 completed
11/1138 completed
12/1138 completed
13/1138 completed
14/1138 completed
15/1138 completed
16/1138 completed
17/1138 completed
18/1138 completed
19/1138 completed
20/1138 completed
21/1138 completed
22/1138 completed
23/1138 completed
24/1138 completed
25/1138 completed
26/1138 completed
27/1138 completed
28/1138 completed
29/1138 completed
30/1138 completed
31/1138 completed
32/1138 completed
33/1138 completed
34/1138 completed
35/1138 completed
36/1138 completed
37/1138 completed
38/1138 completed
39/1138 completed
40/1138 completed
41/1138 completed
42/1138 completed
43/1138 completed
44/1138 completed
45/1138 completed
46/1138 completed
47/1138 completed
48/1138 completed
49/1138 completed
50/1138 completed
51/1138 completed
52/1138 completed
53/1138 completed
54/1138 completed
55/1138 completed
56/1138 completed
5